# Mortgage Rate Enrichment

This notebook enriches the cleaned MLS **sold** and **listing** datasets with the national **30-year fixed mortgage rate** from FRED.

The workflow follows the Week 3 requirement:

1. Load the cleaned sold and listing datasets.
2. Fetch the FRED `MORTGAGE30US` series directly from the public CSV endpoint.
3. Convert weekly mortgage rates into monthly averages.
4. Create a matching `year_month` key from MLS transaction dates.
5. Left-merge the monthly mortgage rate onto both datasets.
6. Validate that the merge worked correctly.
7. Save both enriched datasets as new CSV files.


## 1. Import packages



In [1]:
import pandas as pd
from pathlib import Path

## 2. Set file paths


In [2]:
# Input files
# Update these only if your actual file names are different.
SOLD_FILE = Path("/Users/amyliu/Desktop/IDX/data/generated/sold_cleaned.csv")
LISTINGS_FILE = Path("/Users/amyliu/Desktop/IDX/data/generated/listing_cleaned.csv")

# Output files
SOLD_OUTPUT_FILE = Path("/Users/amyliu/Desktop/IDX/data/generated/sold_combined_residential_with_mortgage_rates.csv")
LISTINGS_OUTPUT_FILE = Path("/Users/amyliu/Desktop/IDX/data/generated/listing_combined_residential_with_mortgage_rates.csv")

## 3. Load the cleaned MLS datasets

The sold dataset should contain `CloseDate`, while the listing dataset should contain `ListingContractDate`. These fields will be used to create the monthly join key.


In [4]:
sold = pd.read_csv(SOLD_FILE, low_memory=False)
listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

print("Sold dataset shape:", sold.shape)
print("Listings dataset shape:", listings.shape)

print("Sold columns preview:")
print(sold.columns[:20].tolist())

print("Listings columns preview:")
print(listings.columns[:20].tolist())

Sold dataset shape: (397603, 67)
Listings dataset shape: (540183, 71)
Sold columns preview:
['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName']
Listings columns preview:
['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName']


## 4. Fetch 30-year fixed mortgage rate data from FRED

The FRED series used here is `MORTGAGE30US`, which provides the national 30-year fixed mortgage rate. The data is pulled directly from FRED's public CSV URL.


In [5]:
fred_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(fred_url)

# Rename columns for easier interpretation
mortgage.columns = ["date", "rate_30yr_fixed"]

# Convert data types
mortgage["date"] = pd.to_datetime(mortgage["date"], errors="coerce")
mortgage["rate_30yr_fixed"] = pd.to_numeric(mortgage["rate_30yr_fixed"], errors="coerce")

# Remove invalid rows
mortgage = mortgage.dropna(subset=["date", "rate_30yr_fixed"])

print("Mortgage data shape:", mortgage.shape)
mortgage.head()

Mortgage data shape: (2874, 2)


,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


## 5. Resample weekly rates to monthly averages

MLS transactions are analyzed by calendar month, so the weekly FRED mortgage rate must be converted to a monthly average before merging.

The monthly key is stored as `year_month`, such as `2026-03`.


In [6]:
mortgage["year_month"] = mortgage["date"].dt.to_period("M")

mortgage_monthly = (
    mortgage
    .groupby("year_month", as_index=False)["rate_30yr_fixed"]
    .mean()
)

mortgage_monthly["rate_30yr_fixed"] = mortgage_monthly["rate_30yr_fixed"].round(3)

print("Monthly mortgage data shape:", mortgage_monthly.shape)
mortgage_monthly.tail()

Monthly mortgage data shape: (661, 2)


,year_month,rate_30yr_fixed
656,2025-12,6.190
657,2026-01,6.102
658,2026-02,6.048
659,2026-03,6.178
660,2026-04,6.340


## 6. Create matching `year_month` keys in MLS datasets

For the sold dataset, the monthly key should be based on `CloseDate` because it represents the transaction close month.

For the listing dataset, the monthly key should be based on `ListingContractDate` because it represents the listing start/contract month.


In [7]:
# Check required date columns
required_sold_date_col = "CloseDate"
required_listing_date_col = "ListingContractDate"

if required_sold_date_col not in sold.columns:
    raise KeyError(f"Sold dataset is missing required column: {required_sold_date_col}")

if required_listing_date_col not in listings.columns:
    raise KeyError(f"Listings dataset is missing required column: {required_listing_date_col}")

# Create monthly keys
sold["CloseDate"] = pd.to_datetime(sold["CloseDate"], errors="coerce")
sold["year_month"] = sold["CloseDate"].dt.to_period("M")

listings["ListingContractDate"] = pd.to_datetime(listings["ListingContractDate"], errors="coerce")
listings["year_month"] = listings["ListingContractDate"].dt.to_period("M")

print("Sold year_month range:", sold["year_month"].min(), "to", sold["year_month"].max())
print("Listings year_month range:", listings["year_month"].min(), "to", listings["year_month"].max())

Sold year_month range: 2024-01 to 2026-03
Listings year_month range: 2024-01 to 2026-03


## 7. Merge monthly mortgage rates into both datasets

I use a **left join** because we want to preserve every MLS record and add the mortgage rate when the matching month exists.


In [9]:
sold_with_rates = sold.merge(
    mortgage_monthly,
    on="year_month",
    how="left"
)

listings_with_rates = listings.merge(
    mortgage_monthly,
    on="year_month",
    how="left"
)

print("Sold shape before merge:", sold.shape)
print("Sold shape after merge:", sold_with_rates.shape)

print("Listings shape before merge:", listings.shape)
print("Listings shape after merge:", listings_with_rates.shape)

Sold shape before merge: (397603, 68)
Sold shape after merge: (397603, 69)
Listings shape before merge: (540183, 72)
Listings shape after merge: (540183, 73)


## 8. Validate the merge

The main validation check is whether any rows have missing `rate_30yr_fixed` after the merge.

A small number of missing values may occur if:

- the MLS date is missing or invalid;
- the MLS date is outside the FRED series date range;
- the wrong date column was used;
- the `year_month` key was not created correctly.


In [13]:
sold_missing_rates = sold_with_rates["rate_30yr_fixed"].isna().sum()
listings_missing_rates = listings_with_rates["rate_30yr_fixed"].isna().sum()

print("Validation Results:")
print(f"Missing mortgage rates in sold dataset: {sold_missing_rates:,}")
print(f"Missing mortgage rates in listings dataset: {listings_missing_rates:,}")

print("Missing rate percentage:")
print(f"Sold: {sold_missing_rates / len(sold_with_rates):.2%}")
print(f"Listings: {listings_missing_rates / len(listings_with_rates):.2%}")

Validation Results:
Missing mortgage rates in sold dataset: 0
Missing mortgage rates in listings dataset: 0
Missing rate percentage:
Sold: 0.00%
Listings: 0.00%


## 9. Inspect unmatched rows, if any

This helps diagnose whether missing mortgage rates are caused by missing dates or out-of-range dates.


In [14]:
if sold_missing_rates > 0:
    print("Sold rows with missing mortgage rates:")
    display(
        sold_with_rates.loc[
            sold_with_rates["rate_30yr_fixed"].isna(),
            ["CloseDate", "year_month", "rate_30yr_fixed"]
        ].head(10)
    )
else:
    print("No missing mortgage rates in sold dataset.")

if listings_missing_rates > 0:
    print("Listings rows with missing mortgage rates:")
    display(
        listings_with_rates.loc[
            listings_with_rates["rate_30yr_fixed"].isna(),
            ["ListingContractDate", "year_month", "rate_30yr_fixed"]
        ].head(10)
    )
else:
    print("No missing mortgage rates in listings dataset.")

No missing mortgage rates in sold dataset.
No missing mortgage rates in listings dataset.


## 10. Preview enriched datasets

The preview confirms that each MLS row now has a monthly mortgage rate attached.


In [15]:
sold_preview_cols = ["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]
sold_preview_cols = [col for col in sold_preview_cols if col in sold_with_rates.columns]

print("Sold preview:")
display(sold_with_rates[sold_preview_cols].head())

listing_preview_cols = ["ListingContractDate", "year_month", "ListPrice", "rate_30yr_fixed"]
listing_preview_cols = [col for col in listing_preview_cols if col in listings_with_rates.columns]

print("Listings preview:")
display(listings_with_rates[listing_preview_cols].head())

Sold preview:


,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.0,6.642
1,2024-01-05,2024-01,815000.0,6.642
2,2024-01-05,2024-01,810000.0,6.642
3,2024-01-30,2024-01,858000.0,6.642
4,2024-01-29,2024-01,1890500.0,6.642


Listings preview:


,ListingContractDate,year_month,ListPrice,rate_30yr_fixed
0,2024-01-01,2024-01,1340000.0,6.642
1,2024-01-24,2024-01,2500000.0,6.642
2,2024-01-12,2024-01,3150000.0,6.642
3,2024-01-20,2024-01,3090000.0,6.642
4,2024-01-12,2024-01,12725000.0,6.642


## 11. Save enriched datasets

Before saving, `year_month` is converted from a Period type to a string so it exports cleanly to CSV.


In [16]:
# Convert year_month to string before exporting
sold_with_rates["year_month"] = sold_with_rates["year_month"].astype(str)
listings_with_rates["year_month"] = listings_with_rates["year_month"].astype(str)

# Make sure output folder exists
SOLD_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
LISTINGS_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

sold_with_rates.to_csv(SOLD_OUTPUT_FILE, index=False)
listings_with_rates.to_csv(LISTINGS_OUTPUT_FILE, index=False)

print("Files saved successfully:")
print("-", SOLD_OUTPUT_FILE)
print("-", LISTINGS_OUTPUT_FILE)

Files saved successfully:
- /Users/amyliu/Desktop/IDX/data/generated/sold_combined_residential_with_mortgage_rates.csv
- /Users/amyliu/Desktop/IDX/data/generated/listing_combined_residential_with_mortgage_rates.csv


## 13. Summary

This notebook successfully completes the mortgage rate enrichment task by:

- fetching the FRED `MORTGAGE30US` series;
- converting weekly mortgage rates to monthly averages;
- creating `year_month` keys from MLS transaction dates;
- merging the monthly mortgage rate into both sold and listing datasets;
- validating whether the merge produced missing mortgage-rate values;
- saving both enriched datasets as new CSV files.
